# CHB-MIT datasets: how to generate and load them

This notebook shows how to build and load the two window datasets produced by `datasets/`.

| Dataset | Script | Output | Intended CV |
|---|---|---|---|
| **Balanced windows** | `datasets/build_balanced_windows_dataset.py` | `datasets/data/balanced_windows/` | Leave-one-patient-out |
| **Recording CV** | `datasets/build_recording_cv_dataset.py` | `datasets/data/recording_cv/win{1,3,5,10}s/per_patient/` | Within-patient k-fold / leave-one-recording-out |

The data is in microvolts, sampled at 256 Hz. Balanced windows have shape `(21 channels, 768 samples)`, i.e. 3 s. The recording-CV dataset comes in four window lengths, 1, 3, 5 and 10 s, which is 256 to 2560 samples. Label `1` means seizure and `0` means normal.
`X` is stored as float16 on disk. The `load_windows` helpers cast it back to float32.

Run this notebook from the `metapaper/` folder so that `import datasets` resolves to the local package.

In [ ]:
import numpy as np

from datasets.data_loader import load_patient_data
from datasets.preprocessing import segments_to_windows, balance_dataset, cap_class_ratio
from datasets import build_balanced_windows_dataset as balanced
from datasets import build_recording_cv_dataset as reccv

print("balanced ->", balanced.OUT_DIR)
print("rec. CV  ->", [str(reccv.per_patient_dir(w)) for w in reccv.WINDOW_SECONDS])

## 1. Generating the data

### 1a. What the pipeline does, step by step (one patient)

`load_patient_data` reads the EDFs and returns **variable-length segments**: one per seizure, one per normal stretch between seizures, and one per fully normal file.
With `return_recordings=True` it also returns the EDF filename each segment came from.

In [ ]:
X_seg, y_seg, rec_seg = load_patient_data("chb01", only_seizure_files=True,
                                          return_recordings=True, progress=False)
print(len(X_seg), "segments;", int((y_seg == 1).sum()), "seizure segments")
print("first segments:", list(zip(rec_seg[:4], y_seg[:4])))

`segments_to_windows` cuts each segment into fixed-size windows, and each window inherits its segment's label.
If you pass `groups=`, the recording id is carried over to each window as well.

In [ ]:
X, y, rec = segments_to_windows(X_seg, y_seg, window_samples=256 * 3, groups=rec_seg)
print(X.shape, "seizure:", int((y == 1).sum()), "normal:", int((y == 0).sum()))

Then the class balance is set in one of two ways:

- **Balanced dataset:** `balance_dataset` undersamples to 1:1 over the whole patient.
- **Recording-CV dataset:** `cap_class_ratio(max_ratio=5)` is applied **per recording**, keeping at most 5 normal windows per seizure window.

In [ ]:
Xb, yb = balance_dataset(X, y, method="undersample", seed=0)
print("1:1 balanced :", Xb.shape, np.bincount(yb))

Xc, yc, recc = cap_class_ratio(X, y, rec, max_ratio=5, seed=0)
print("1:5 capped   :", Xc.shape, np.bincount(yc))

`process_patient` in each build script runs this whole chain for one patient and returns the arrays that get saved.
In the recording-CV script it takes `window_sec`, and also returns `dropped`: recordings whose seizures are all shorter than the window and so give no seizure window.
Read the EDFs once with `load_segments` and pass them in as `segments=` to window the same data at several lengths.

In [ ]:
segments = reccv.load_segments("chb01")                 # read EDFs once
for w in reccv.WINDOW_SECONDS:
    X, y, rec, dropped = reccv.process_patient("chb01", window_sec=w, segments=segments)
    print(f"{w:2d}s: {X.shape}  labels={np.bincount(y)}  recordings={len(np.unique(rec))}  dropped={dropped}")

### 1b. Building the full datasets (all 24 patients)

Each build overwrites its folder in `datasets/data/`. The balanced build reads every EDF (several minutes). The recording-CV build reads only the recordings with seizures, then writes all four window lengths.
Use `--windows` to build only some of the lengths.
Set the flags to `True` to rebuild. The same builds can be run from a terminal:

```bash
python datasets/build_balanced_windows_dataset.py
python datasets/build_recording_cv_dataset.py               # 1, 3, 5, 10 s
python datasets/build_recording_cv_dataset.py --windows 5   # only 5 s
```

In [ ]:
REBUILD_BALANCED = False
REBUILD_RECORDING_CV = False

if REBUILD_BALANCED:
    balanced.main()
if REBUILD_RECORDING_CV:
    reccv.main()                      # or reccv.main([5, 10])

## 2. Loading the balanced dataset (leave-one-patient-out)

The combined file holds every patient, with `patient_id` (an index into `patients`) on every window.

In [ ]:
d = balanced.load_windows(balanced.OUT_DIR / "chbmit_windows_all.npz")
X, y, pid, patients = d["X"], d["y"], d["patient_id"], d["patients"]
print(X.shape, X.dtype, "labels:", np.bincount(y))
print("patients:", len(patients), "->", patients[:3], "...")

A leave-one-patient-out loop: each fold holds out one patient. Scikit-learn's `LeaveOneGroupOut` with `groups=pid` produces the same splits.

In [ ]:
for test_p in np.unique(pid)[:3]:            # first 3 folds only, for brevity
    train_idx = np.flatnonzero(pid != test_p)
    test_idx = np.flatnonzero(pid == test_p)
    print(f"test={patients[test_p]}  train={len(train_idx):5d}  test={len(test_idx):4d}")

A single patient can also be loaded from `per_patient/`:

In [ ]:
d = balanced.load_windows(balanced.OUT_DIR / "per_patient" / "chb01.npz")
print(d["X"].shape, np.bincount(d["y"]))

## 3. Loading the recording-CV dataset (within-patient)

Pick a window length with `per_patient_dir(window_sec)`. Each patient file has `X`, `y`, `recording` (EDF filename for each window), `window_len` (in samples) and `window_sec`.

In [ ]:
WINDOW_SEC = 3                                        # 1, 3, 5 or 10
d = reccv.load_windows(reccv.per_patient_dir(WINDOW_SEC) / "chb01.npz")
X, y, rec = d["X"], d["y"], d["recording"]
print(X.shape, "labels:", np.bincount(y))
for r in np.unique(rec):
    m = rec == r
    print(f"  {r}: seizure={int((y[m] == 1).sum()):3d}  normal={int((y[m] == 0).sum()):3d}")

`recording_folds` yields `(train_idx, test_idx)` pairs in which no recording is in both sets:

- `n_splits=5` or `10` uses `StratifiedGroupKFold`.
- `n_splits=None` gives leave-one-recording-out.
- If a patient has fewer recordings than `n_splits`, it falls back to leave-one-recording-out and warns.

Balance **only the training fold**, so the test fold keeps the real 1:5 ratio.

In [ ]:
for k, (train_idx, test_idx) in enumerate(reccv.recording_folds(y, rec, n_splits=5, seed=0)):
    assert not set(rec[train_idx]) & set(rec[test_idx])       # no recording leakage
    X_tr, y_tr = balance_dataset(X[train_idx], y[train_idx], seed=k)
    X_te, y_te = X[test_idx], y[test_idx]
    print(f"fold {k}: test recs={sorted(map(str, set(rec[test_idx])))}  "
          f"train(bal)={np.bincount(y_tr)}  test={np.bincount(y_te)}")

Leave-one-recording-out (the number of folds equals the number of recordings):

In [ ]:
folds = list(reccv.recording_folds(y, rec, n_splits=None))
print(len(folds), "folds;", "held-out:", [str(rec[te][0]) for _, te in folds])

How many recordings each patient has at this window length, which limits how many folds are possible:

In [ ]:
for f in sorted(reccv.per_patient_dir(WINDOW_SEC).glob("chb*.npz")):
    d = np.load(f)
    n_rec = len(np.unique(d["recording"]))
    note = "10-fold ok" if n_rec >= 10 else "5-fold ok" if n_rec >= 5 else "LORO only"
    print(f"{f.stem}: {len(d['y']):5d} windows, {n_rec:2d} recordings  ({note})")

Comparing the four window lengths for one patient:

In [ ]:
for w in reccv.WINDOW_SECONDS:
    d = reccv.load_windows(reccv.per_patient_dir(w) / "chb01.npz")
    print(f"{w:2d}s: X={d['X'].shape}  labels={np.bincount(d['y'])}  "
          f"recordings={len(np.unique(d['recording']))}")

Converting to PyTorch tensors, if needed:

```python
import torch
X_t = torch.from_numpy(X_tr)                  # (n, 21, 768) float32
y_t = torch.from_numpy(y_tr.astype(np.int64))
```